In [1]:
import pandas as pd
import numpy as np
from statsmodels.stats.inter_rater import fleiss_kappa

FOLDER_NAME = 'annotations'

FILE_NAMES = [
    '001_Adam',
    #'002_Gianfranco',
    '003_Javier',
    '004_Marino',   
]

SHEET_NAMES = [
    'CSC',
    'MultiPICo',
]

ACCEPTABLE_LABELS = ['non-triggering', 'triggering']

In [2]:
label_to_index = {label: i for i, label in enumerate(ACCEPTABLE_LABELS)}
label_to_index

{'non-triggering': 0, 'triggering': 1}

In [3]:
# load dfs
dfs = {}
for file_name in FILE_NAMES:
    for sheet_name in SHEET_NAMES:
        df = pd.read_excel(f'{FOLDER_NAME}/{file_name}.xlsx', sheet_name=sheet_name)
        df.rename(columns={
            'Given this conversational context, is an ironic response likely or natural?': 'label'
        }, inplace=True)
        dfs[f'{file_name}_{sheet_name}'] = df

print(dfs.keys())
#dfs['001_Adam_CSC'].head()

dict_keys(['001_Adam_CSC', '001_Adam_MultiPICo', '003_Javier_CSC', '003_Javier_MultiPICo', '004_Marino_CSC', '004_Marino_MultiPICo'])


In [4]:
# remove invalid labels

ids_to_remove = set()
for key, df in dfs.items():
    invalid_ids = df.loc[~df['label'].isin(ACCEPTABLE_LABELS), 'unique_id']
    ids_to_remove.update(invalid_ids)

print(len(ids_to_remove))
print(ids_to_remove)

20
{'10302135371128027311_12495562528796468539', '109_1010', '15377547143975358910_1711386745880565388', '208_2036', '248_2078', '11804440989245452213_8697572573754267076', '17260434174864013858_11445425254358103606', '15735844402404738656_4292779336955660346', '8648986929715964531_6080813350745733225', '13867510771314669985_3692101571900816325', '8755515661243036834_15818415332632239449', '13351787454385947951_4930900043121476558', '7199594443675332653_13942175382689258523', '209_2038', '17565001203181101696_9217991978256712534', '12504922140387146689_17713436929031669011', '1817886739073302709_8931910327729403976', '17684334533513677021_15921899341881086227', '266_2054', '111_1036'}


In [5]:
# filter dfs
for key in dfs.keys():
    dfs[key] = dfs[key][~dfs[key]['unique_id'].isin(ids_to_remove)]
    print(key, len(dfs[key]))

001_Adam_CSC 94
001_Adam_MultiPICo 86
003_Javier_CSC 94
003_Javier_MultiPICo 86
004_Marino_CSC 94
004_Marino_MultiPICo 86


In [6]:
# calcolo Fleiss
for sheet_name in SHEET_NAMES:
    print(f'\n=== {sheet_name} ===')

    # prendi solo annotatori di questo sheet
    sheet_dfs = [dfs[key] for key in dfs if sheet_name in key]

    # 🔹 intersezione degli ID (IMPORTANTISSIMO)
    common_ids = set(sheet_dfs[0]['unique_id'])
    for df in sheet_dfs[1:]:
        common_ids &= set(df['unique_id'])
    common_ids = sorted(common_ids)

    matrix = []

    for uid in common_ids:
        row = [0] * len(ACCEPTABLE_LABELS)

        for df in sheet_dfs:
            label = df.loc[df['unique_id'] == uid, 'label'].values[0]
            row[label_to_index[label]] += 1

        matrix.append(row)

    matrix = np.array(matrix)

    # 🔹 controllo: ogni riga deve sommare a 4 annotatori
    assert np.all(matrix.sum(axis=1) == len(sheet_dfs)), "Errore: numero annotatori non costante"

    kappa = fleiss_kappa(matrix)
    print('Fleiss kappa tra tutti gli annotatori:', kappa)



=== CSC ===
Fleiss kappa tra tutti gli annotatori: 0.5582706766917295

=== MultiPICo ===
Fleiss kappa tra tutti gli annotatori: 0.37430709534368084


In [7]:
print(f'\n=== CSC + MultiPICo (uniti) ===')

# Per ogni annotatore, concatena i due sheet in un unico df
combined_dfs = {}
for file_name in FILE_NAMES:
    parts = [dfs[f'{file_name}_{sheet_name}'] for sheet_name in SHEET_NAMES]
    combined_dfs[file_name] = pd.concat(parts, ignore_index=True)

# Intersezione degli unique_id comuni a tutti gli annotatori
all_combined = list(combined_dfs.values())
common_ids_combined = set(all_combined[0]['unique_id'])
for df in all_combined[1:]:
    common_ids_combined &= set(df['unique_id'])
common_ids_combined = sorted(common_ids_combined)

print(f'Items comuni (entrambi i sheet): {len(common_ids_combined)}')

# Costruzione matrice Fleiss
matrix_combined = []
for uid in common_ids_combined:
    row = [0] * len(ACCEPTABLE_LABELS)
    for df in all_combined:
        label = df.loc[df['unique_id'] == uid, 'label'].values[0]
        row[label_to_index[label]] += 1
    matrix_combined.append(row)

matrix_combined = np.array(matrix_combined)
assert np.all(matrix_combined.sum(axis=1) == len(FILE_NAMES)), "Errore: numero annotatori non costante"

kappa_combined = fleiss_kappa(matrix_combined)
print('Fleiss kappa (sheet uniti):', kappa_combined)


=== CSC + MultiPICo (uniti) ===
Items comuni (entrambi i sheet): 180
Fleiss kappa (sheet uniti): 0.4794138585666823


In [8]:
from itertools import combinations

# Calcolo Fleiss kappa per tutti i gruppi di 3 annotatori
for sheet_name in SHEET_NAMES:
    print(f'\n{"="*40}')
    print(f'Sheet: {sheet_name}')
    print(f'{"="*40}')

    annotators = [f for f in FILE_NAMES]
    
    for trio in combinations(annotators, 3):
        keys = [f'{name}_{sheet_name}' for name in trio]
        trio_dfs = [dfs[k] for k in keys]
        
        # Intersezione degli ID comuni ai tre annotatori
        common_ids = set(trio_dfs[0]['unique_id'])
        for df in trio_dfs[1:]:
            common_ids &= set(df['unique_id'])
        common_ids = sorted(common_ids)
        
        if not common_ids:
            print(f'\n{trio}: nessun ID comune, skip.')
            continue
        
        # Costruzione matrice Fleiss (n_items × n_categorie)
        matrix = []
        for uid in common_ids:
            row = [0] * len(ACCEPTABLE_LABELS)
            for df in trio_dfs:
                label = df.loc[df['unique_id'] == uid, 'label'].values[0]
                row[label_to_index[label]] += 1
            matrix.append(row)
        
        matrix = np.array(matrix)
        assert np.all(matrix.sum(axis=1) == 3), "Errore: ogni riga deve sommare a 3"
        
        kappa = fleiss_kappa(matrix)
        names = ' / '.join(n.split('_', 1)[1] for n in trio)  # es. "Adam / Gianfranco / Javier"
        print(f'\n  Trio: {names}')
        print(f'  Items comuni: {len(common_ids)}')
        print(f'  Fleiss kappa: {kappa:.4f}')


Sheet: CSC

  Trio: Adam / Javier / Marino
  Items comuni: 94
  Fleiss kappa: 0.5583

Sheet: MultiPICo

  Trio: Adam / Javier / Marino
  Items comuni: 86
  Fleiss kappa: 0.3743


In [9]:
import pandas as pd

# lista delle label accettabili
ACCEPTABLE_LABELS = ['triggering', 'non-triggering']
label_to_index = {label: i for i, label in enumerate(ACCEPTABLE_LABELS)}

gold_labels = {}

for sheet_name in SHEET_NAMES:
    print(f'\n=== {sheet_name} ===')

    # prendi solo annotatori di questo sheet
    sheet_dfs = [dfs[key] for key in dfs if sheet_name in key]

    # intersezione degli ID
    common_ids = set(sheet_dfs[0]['unique_id'])
    for df in sheet_dfs[1:]:
        common_ids &= set(df['unique_id'])
    common_ids = sorted(common_ids)

    for uid in common_ids:
        label_counts = [0] * len(ACCEPTABLE_LABELS)

        for df in sheet_dfs:
            label = df.loc[df['unique_id'] == uid, 'label'].values[0]
            label_counts[label_to_index[label]] += 1

        # voto maggioritario (tie-break → 'triggering')
        if label_counts[label_to_index['non-triggering']] > label_counts[label_to_index['triggering']]:
            gold_labels[uid] = '0'
        else:
            gold_labels[uid] = '1'

# 🔹 crea DataFrame
gold_df = pd.DataFrame({
    'unique_id': list(gold_labels.keys()),
    'gold_label': list(gold_labels.values())
})

# 🔹 salva CSV
gold_df.to_csv('annotations/gold_labels.csv', index=False)
print("CSV gold_labels.csv creato correttamente!")


=== CSC ===

=== MultiPICo ===
CSV gold_labels.csv creato correttamente!


In [11]:
import json
import pandas as pd
from scipy.stats import spearmanr, pearsonr
from sklearn.metrics import precision_score, recall_score, f1_score

# ── paths ───────────────────────────────────────────────────────────────
GOLD_CSV = 'annotations/gold_labels.csv'
CSC_JSON = '../datasets/CSC_meta-llama_Llama-3.1-8B-Instruct_n_turn_2_filtered.json'
MULTIPICO_JSON = '../datasets/MultiPICo_meta-llama_Llama-3.1-8B-Instruct_n_turn_2_filtered.json'

# ── load gold labels ────────────────────────────────────────────────────
gold_df = pd.read_csv(GOLD_CSV)
gold_df['gold_label'] = gold_df['gold_label'].astype(int)

# ── helper: estrai (unique_id, label) dal JSON ─────────────────────────
def load_dataset_labels(json_path, label_field):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    rows = []
    for entry in data['entries']:
        uid = entry['unique_id']
        lbl = entry['annotations'].get(label_field)
        rows.append({'unique_id': uid, 'dataset_label': lbl})
    return pd.DataFrame(rows)

# ── carica dataset ─────────────────────────────────────────────────────
csc_df       = load_dataset_labels(CSC_JSON, 'sarcasm_appropriateness_label')
multipico_df = load_dataset_labels(MULTIPICO_JSON, 'irony_appropriateness_label')
dataset_df   = pd.concat([csc_df, multipico_df], ignore_index=True)

# ── merge con gold labels ──────────────────────────────────────────────
merged = gold_df.merge(dataset_df, on='unique_id', how='inner').dropna(subset=['dataset_label'])
merged['dataset_label'] = merged['dataset_label'].astype(int)

print(f"Entries after merge & dropna: {len(merged)}")
print(merged[['unique_id', 'gold_label', 'dataset_label']].head())

# ── funzioni di valutazione ──────────────────────────────────────────────
def accuracy(y_true, y_pred):
    return (y_true == y_pred).mean()

Entries after merge & dropna: 180
  unique_id  gold_label  dataset_label
0  101_1015           1              1
1  103_1014           0              1
2  103_1019           1              1
3  103_1031           1              1
4  103_1034           1              1


In [12]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr, pearsonr
from statsmodels.stats.inter_rater import fleiss_kappa

ACCEPTABLE_LABELS = ['triggering', 'non-triggering']
label_to_index = {label: i for i, label in enumerate(ACCEPTABLE_LABELS)}

# ── COSTRUZIONE GOLD LABELS ───────────────────────────────────────────────────
# Per ogni sheet calcola:
#   - gold label per voto maggioritario (tutti gli item)
#   - flag "unanime" (tutti e 3 gli annotatori concordano)

gold_records = {}  # uid -> {'gold': '0'/'1', 'unanimous': bool}

for sheet_name in SHEET_NAMES:
    sheet_dfs = [dfs[key] for key in dfs if sheet_name in key]
    n_annotators = len(sheet_dfs)

    common_ids = set(sheet_dfs[0]['unique_id'])
    for df in sheet_dfs[1:]:
        common_ids &= set(df['unique_id'])
    common_ids = sorted(common_ids)

    for uid in common_ids:
        label_counts = [0] * len(ACCEPTABLE_LABELS)
        for df in sheet_dfs:
            label = df.loc[df['unique_id'] == uid, 'label'].values[0]
            label_counts[label_to_index[label]] += 1

        is_unanimous = max(label_counts) == n_annotators

        if label_counts[label_to_index['non-triggering']] > label_counts[label_to_index['triggering']]:
            gold = '0'
        else:
            gold = '1'

        gold_records[uid] = {'gold': gold, 'unanimous': is_unanimous}

gold_df = pd.DataFrame([
    {'unique_id': uid, 'gold_label': v['gold'], 'unanimous': v['unanimous']}
    for uid, v in gold_records.items()
])

print(f"Item totali:    {len(gold_df)}")
print(f"Item unanimi:   {gold_df['unanimous'].sum()}")
print(f"Item non unani: {(~gold_df['unanimous']).sum()}")

gold_df.to_csv('annotations/gold_labels.csv', index=False)


from sklearn.metrics import cohen_kappa_score
# pip install krippendorff  (se non installato)
import krippendorff

def print_metrics(df, label=''):
    y_gold    = df['gold_label'].astype(int).values
    y_dataset = df['dataset_label'].astype(int).values
    n = len(df)

    acc = (y_gold == y_dataset).mean()
    tp  = ((y_gold == 1) & (y_dataset == 1)).sum()
    fp  = ((y_gold == 0) & (y_dataset == 1)).sum()
    fn  = ((y_gold == 1) & (y_dataset == 0)).sum()
    prec   = tp / (tp + fp) if (tp + fp) > 0 else float('nan')
    recall = tp / (tp + fn) if (tp + fn) > 0 else float('nan')
    f1     = 2 * prec * recall / (prec + recall) if (prec + recall) > 0 else float('nan')
    rho, p_s = spearmanr(y_gold, y_dataset)

    # Cohen κ — per 2 rater, label nominali: è la metrica giusta qui
    ck = cohen_kappa_score(y_gold, y_dataset)

    # Krippendorff α — più usato in NLP/linguistica, robusto ai dati mancanti
    # vuole matrice rater × item
    ka = krippendorff.alpha(
        reliability_data=np.array([y_gold, y_dataset]),
        level_of_measurement='nominal'
    )

    print(f'\n── {label} (n={n}) ──')
    print(f'  Accuracy       : {acc:.4f}')
    print(f'  Precision (1)  : {prec:.4f}')
    print(f'  Recall    (1)  : {recall:.4f}')
    print(f'  F1        (1)  : {f1:.4f}')
    print(f'  Cohen κ        : {ck:.4f}')
    print(f'  Krippendorff α : {ka:.4f}')
    print(f'  Spearman       : ρ = {rho:.4f},  p = {p_s:.4e}')


# ── MERGE CON DATASET LABELS ──────────────────────────────────────────────────
# (adatta questa parte al tuo merge esistente)
merged = gold_df.merge(dataset_df, on='unique_id').dropna(subset=['gold_label', 'dataset_label'])
merged_csc      = merged[merged['unique_id'].str.startswith('1')]   # adatta al tuo filtro
merged_multipico = merged[merged['unique_id'].str.startswith('2')]  # adatta al tuo filtro


# ── RISULTATI: tutti gli item ─────────────────────────────────────────────────
print('\n' + '='*55)
print('CON TUTTI GLI ITEM (voto maggioritario)')
print('='*55)
print_metrics(merged,          'Global')
print_metrics(merged_csc,      'CSC')
print_metrics(merged_multipico,'MultiPICo')


# ── RISULTATI: solo item unanimi ──────────────────────────────────────────────
print('\n' + '='*55)
print('SOLO ITEM UNANIMI (tutti e 3 gli annotatori concordano)')
print('='*55)
merged_unani         = merged[merged['unanimous'] == True]
merged_csc_unani     = merged_csc[merged_csc['unanimous'] == True]
merged_multi_unani   = merged_multipico[merged_multipico['unanimous'] == True]

print_metrics(merged_unani,      f'Global unanimi  ({len(merged_unani)}/{len(merged)} item)')
print_metrics(merged_csc_unani,  f'CSC unanimi     ({len(merged_csc_unani)}/{len(merged_csc)} item)')
print_metrics(merged_multi_unani,f'MultiPICo unani ({len(merged_multi_unani)}/{len(merged_multipico)} item)')

Item totali:    180
Item unanimi:   115
Item non unani: 65

CON TUTTI GLI ITEM (voto maggioritario)

── Global (n=180) ──
  Accuracy       : 0.6000
  Precision (1)  : 0.4157
  Recall    (1)  : 0.6491
  F1        (1)  : 0.5068
  Cohen κ        : 0.1967
  Krippendorff α : 0.1727
  Spearman       : ρ = 0.2106,  p = 4.5427e-03

── CSC (n=81) ──
  Accuracy       : 0.5679
  Precision (1)  : 0.4000
  Recall    (1)  : 0.6923
  F1        (1)  : 0.5070
  Cohen κ        : 0.1689
  Krippendorff α : 0.1278
  Spearman       : ρ = 0.1892,  p = 9.0678e-02

── MultiPICo (n=65) ──
  Accuracy       : 0.6462
  Precision (1)  : 0.5172
  Recall    (1)  : 0.6250
  F1        (1)  : 0.5660
  Cohen κ        : 0.2718
  Krippendorff α : 0.2730
  Spearman       : ρ = 0.2753,  p = 2.6467e-02

SOLO ITEM UNANIMI (tutti e 3 gli annotatori concordano)

── Global unanimi  (115/180 item) (n=115) ──
  Accuracy       : 0.6522
  Precision (1)  : 0.4706
  Recall    (1)  : 0.6486
  F1        (1)  : 0.5455
  Cohen κ        : 0

In [13]:
import numpy as np
from sklearn.metrics import (cohen_kappa_score, f1_score,
                             precision_score, recall_score, accuracy_score)
from scipy.stats import spearmanr

N_RUNS = 1000
SEED   = 42

def print_comparison(df, label=''):
    np.random.seed(SEED)

    y_gold    = df['gold_label'].astype(int).values    # annotatori umani = y_true
    y_dataset = df['dataset_label'].astype(int).values # label dataset    = y_pred
    n = len(df)
    p = y_dataset.mean()  # prevalenza del dataset, usata per il random

    # ── metriche reali (y_gold = riferimento, y_dataset = predizione) ─────────
    acc     = accuracy_score(y_gold, y_dataset)
    prec    = precision_score(y_gold, y_dataset, pos_label=1, zero_division=0)
    recall  = recall_score(y_gold, y_dataset, pos_label=1, zero_division=0)
    f1_real = f1_score(y_gold, y_dataset, pos_label=1, zero_division=0)
    ck      = cohen_kappa_score(y_gold, y_dataset)
    rho, _  = spearmanr(y_gold, y_dataset)

    # ── baseline random(p) — logica CANTRIP ──────────────────────────────────
    # y_rand imita la distribuzione del dataset (stessa prevalenza)
    # confrontato contro y_gold (gli annotatori umani), come y_dataset
    rand_metrics = {k: [] for k in ['acc', 'prec', 'rec', 'f1', 'ck', 'rho']}

    for _ in range(N_RUNS):
        y_rand = np.random.binomial(1, p, size=n)  # scalare p, non array

        rand_metrics['acc'].append(accuracy_score(y_gold, y_rand))
        rand_metrics['prec'].append(precision_score(y_gold, y_rand, pos_label=1, zero_division=0))
        rand_metrics['rec'].append(recall_score(y_gold, y_rand, pos_label=1, zero_division=0))
        rand_metrics['f1'].append(f1_score(y_gold, y_rand, pos_label=1, zero_division=0))
        rand_metrics['ck'].append(cohen_kappa_score(y_gold, y_rand))
        rand_metrics['rho'].append(spearmanr(y_gold, y_rand)[0])

    # ── stampa tabella ────────────────────────────────────────────────────────
    print(f'\n{"─"*60}')
    print(f'  {label} (n={n})')
    print(f'  prevalenza dataset={p:.1%}  prevalenza gold={y_gold.mean():.1%}')
    print(f'{"─"*60}')
    print(f'  {"Metrica":<18} {"Reale":>10}   {"Random(p)":>12}   {"Δ":>8}')
    print(f'  {"─"*18}   {"─"*10}   {"─"*12}   {"─"*8}')

    rows = [
        ('Accuracy',      acc,     rand_metrics['acc']),
        ('Precision (1)', prec,    rand_metrics['prec']),
        ('Recall (1)',    recall,  rand_metrics['rec']),
        ('F1 (1)',        f1_real, rand_metrics['f1']),
        ('Cohen κ',       ck,      rand_metrics['ck']),
        ('Spearman ρ',    rho,     rand_metrics['rho']),
    ]
    for name, real, rand_vals in rows:
        r_mean = np.nanmean(rand_vals)
        r_std  = np.nanstd(rand_vals)
        delta  = real - r_mean
        sign   = '+' if delta >= 0 else ''
        print(f'  {name:<18} {real:>10.3f}   {r_mean:.3f}±{r_std:.3f}   {sign}{delta:.3f}')


# ── output ────────────────────────────────────────────────────────────────────
print('\n' + '='*60)
print('CONFRONTO REALE vs RANDOM(p) BASELINE')
print('='*60)
print_comparison(merged,           'Global — tutti gli item')
print_comparison(merged_csc,       'CSC — tutti gli item')
print_comparison(merged_multipico, 'MultiPICo — tutti gli item')

print('\n' + '='*60)
print('CONFRONTO REALE vs RANDOM(p) BASELINE — solo unanimi')
print('='*60)
print_comparison(merged_unani,       'Global unanimi')
print_comparison(merged_csc_unani,   'CSC unanimi')
print_comparison(merged_multi_unani, 'MultiPICo unanimi')


CONFRONTO REALE vs RANDOM(p) BASELINE

────────────────────────────────────────────────────────────
  Global — tutti gli item (n=180)
  prevalenza dataset=49.4%  prevalenza gold=31.7%
────────────────────────────────────────────────────────────
  Metrica                 Reale      Random(p)          Δ
  ──────────────────   ──────────   ────────────   ────────
  Accuracy                0.600   0.501±0.038   +0.099
  Precision (1)           0.416   0.316±0.036   +0.100
  Recall (1)              0.649   0.494±0.068   +0.155
  F1 (1)                  0.507   0.385±0.045   +0.122
  Cohen κ                 0.197   -0.001±0.071   +0.198
  Spearman ρ              0.211   -0.001±0.076   +0.212

────────────────────────────────────────────────────────────
  CSC — tutti gli item (n=81)
  prevalenza dataset=55.6%  prevalenza gold=32.1%
────────────────────────────────────────────────────────────
  Metrica                 Reale      Random(p)          Δ
  ──────────────────   ──────────   ───────